# Domains

In the [Algorithms notebook](https://saminulh.github.io/dpstats/library/tutorials/algorithms/), we used a domain to state the public bounds available to an algorithm. A [`Domain`](https://saminulh.github.io/dpstats/library/reference/domains/dpstats.domains.Domain/) describes the possible values of a single sample, including its number of coordinates.

Here we walk through how to:

1. Declare a domain and check which inputs it contains.
2. Use that domain in a task's metadata.
3. Inspect the geometry an algorithm uses to determine its noise scale.


## Defining bounds and checking membership

A [`BoxDomain`](https://saminulh.github.io/dpstats/library/reference/domains/dpstats.domains.BoxDomain/) supplies a lower and upper bound for each coordinate. Each row of its `bounds` array is one coordinate's interval.

Below, the first coordinate lies in $[0, 1]$ and the second in $[-2, 2]$. The [`contains`](https://saminulh.github.io/dpstats/library/reference/domains/dpstats.domains.Domain/#dpstats.domains.Domain.contains) method returns one Boolean per input row. All four rows of `X` satisfy these bounds; the additional point fails because its first coordinate is greater than 1.


In [ ]:
import numpy as np

from dpstats.domains import BoxDomain

box = BoxDomain(bounds=np.array([[0.0, 1.0], [-2.0, 2.0]]))
X = np.array([[0.1, -1.0], [0.3, 0.0], [0.6, 1.0], [0.8, 0.5]])
print("dimension:", box.dim)
print("contains X:", box.contains(X))
print("contains [1.2, 0.0]:", box.contains(np.array([[1.2, 0.0]])))


## Using a domain in metadata

The domain belongs in the task's metadata because its constraints are public information. We supply the box as an assumption about possible values, independently of the private sample.

For [mean estimation](https://saminulh.github.io/dpstats/mean_estimation/), we combine the domain with the declared number of samples. The metadata's `validate_input` method checks the input's row count, dimensions, and membership in the domain. This input passes those checks, so the cell finishes and prints the metadata.


In [ ]:
from dpstats.tasks.mean_estimation import (
    MeanEstimationInput,
    MeanEstimationMetadata,
)

metadata = MeanEstimationMetadata(n_samples=len(X), domain=box)
input = MeanEstimationInput(X=X)
metadata.validate_input(input)
print(metadata)


## Inspecting the domain's geometry

Algorithms can ask a domain for geometric properties as well as membership. The [`l2_enclosing_ball`](https://saminulh.github.io/dpstats/library/reference/domains/dpstats.domains.Domain/#dpstats.domains.Domain.l2_enclosing_ball) method returns a public center and radius for a Euclidean ball containing the domain.

The Gaussian example in Algorithms used this radius to determine its noise scale. Here we use the same [accounting helper](https://saminulh.github.io/dpstats/library/reference/budgets/dpstats.budgets.accounting.calibrate_gaussian_scale/) to inspect that scale directly. The center and radius below come from the declared box.


In [ ]:
from dpstats.budgets import ApproxDPBudget, accounting

budget = ApproxDPBudget(eps=1.0, delta=1e-6)
center, radius = metadata.domain.l2_enclosing_ball()
sensitivity = radius / metadata.n_samples
sigma = accounting.calibrate_gaussian_scale(budget, sensitivity)
print("center:", center)
print("radius:", radius)
print("noise scale:", sigma)


### Impact of domains on algorithm performance

Even with the same algorithm and privacy budget, the choice of domain can affect performance: tighter bounds can improve accuracy by reducing the required noise. In the following example, we widen the box domain, which necessarily still contains `X`, but we see that as a result, the noise increased. Thus, if public bounds are loose, it may be advantageous to spend some privacy budget estimate tighter bounds, we see an example of this in the [Transformations notebook](https://saminulh.github.io/dpstats/library/tutorials/extend_with_transformations/).

In [ ]:
wider_box = BoxDomain(bounds=np.array([[-1.0, 2.0], [-4.0, 4.0]]))
print("wider box contains X:", wider_box.contains(X))
_, wider_radius = wider_box.l2_enclosing_ball()
wider_sigma = accounting.calibrate_gaussian_scale(
    budget, wider_radius / metadata.n_samples
)
print("original noise scale:", sigma)
print("wider noise scale:", wider_sigma)


## Finding other domains

The [Domains API](https://saminulh.github.io/dpstats/library/reference/domains/) lists the available domain types, including balls, unbounded spaces, categorical blocks, and combinations of domains. Use its descriptions to choose a type, then follow the class link for its constructor fields and examples.

These types share the `Domain` interface used above. A class page's `Domain` base link leads to inherited methods that are not listed on that page. The same membership and metadata checks apply once the domain is constructed.


## Next steps

The [Privacy budgets and accounting notebook](https://saminulh.github.io/dpstats/library/tutorials/privacy_budgets_and_accounting/) goes through the available budget types, noise calibration, and accounting for algorithms with multiple private steps.
